Code for building file manifests for MRI & MEG data.

#### **NOTE: Requires 'MEG_parameter_index.csv' FILE to be in main output/project directory FIRST**

- Also, we've vetted/validated MEG parameters & meta-data at an earlier stage ('audit_MEG_metadata' script), so we can accept all MEG files listed in the 'PARAMETER_INDEX' dataframe here as valid for processing.
- The only "further" filtering we want to do is selection-based, e.g. we might decide to drop any subjects who don't have valid data for ALL selected time-points, etc... 

**Summary:** This script takes the 'master data catalogue' table (one row per subject, with many “MRI_” and “MEG_” filename columns) and turns it into the concrete “shopping list” that the pipeline will actually run: i.e. which subjects are included; which MRI file each included subject will use; and which MEG files (sessions) each included subject will contribute. It then exports two manifests: (1) a subject-level manifest ('subject_manifest.csv'; one row per subject, with MRI + MEG columns) and (2) a MEG file-level manifest ('MEG_manifest.csv'; one row per MEG run). In other words: this script operationalizes the "analysis plan" as set forth in the config.yaml file (group selection + MRI selection + MEG selection + completeness rules) into deterministic CSVs that downstream scripts can trust, without re-deriving selection logic later.

[Runtime: Negligible / very short]

-------

In [ ]:
### LOAD CONFIG.YAML:

import yaml, os
from pathlib import Path
import subprocess

CONFIG_PATH = Path.cwd() / "config.yaml"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"Config file not found: {CONFIG_PATH.resolve()}")

with open(CONFIG_PATH, 'r') as config_file:
    config = yaml.safe_load(config_file)

# config

In [ ]:
### IMPORTS:
import os, re
import datetime
import pandas as pd
from pathlib import Path
import shutil
import numpy as np

In [ ]:
### SET PARAMETERS:

MANIFEST_OVERWRITE = config['manifest_overwrite']
HARD_STOP = config['hard_errors']

HAS_QC = not(config['skip_catalogue_QC']) # Should eval to true if QC stage was *not* skipped
if HAS_QC:
    GOOD_LABELS = config['QC_pass_labels']

# Analysis parameters:
CONDITIONS = config['CONDITIONS']
MRI_SELECTION  = config['MRI_selection']
MEG_SELECTION = config['MEG_selection']
MEG_VALID_PRESETS = ['all'] # QC-based or chronology-based preset options not available for MEG pipeline (only fMRI currently); can still select session_ID types manually though

COMPLETE_SUBS_ONLY = config['complete_MEG_data_only']
EXCLUDE_IF_MISSING = config['exclude_sub_if_missing']

EXPORT_SUBSET = config['save_subset_catalogue']

# Type checks (if needed):
if type(MRI_SELECTION) != str:
    raise Exception("ERROR: 'MRI_selection' parameter in config.yaml must be one plain STRING (either a valid preset code or a study time-point).")

# Other params:
TIME_POINTS_DICT = config['session_ID_mappings'] # Needed for chronological session_ID code ordering
# Sanity-check / guard:
if not isinstance(TIME_POINTS_DICT, dict) or not TIME_POINTS_DICT:
    raise ValueError(
        "\n*******************\n'session_ID_mappings' CONFIG field MUST be set to a NON-EMPTY DICTIONARY \n"
        "defining chronological session order (even if no remapping is desired)."
        "\n\n   --> The minimal possible setting, if no re-mapping is desired, is to set this field to:" \
        "\n           session_ID_mappings:" \
        "\n             UNKNOWN:" \
        "\n               - null")

# Check to make sure 'EXCLUDE_IF_MISSING' contains only time-points that we are actually selecting for:
assert (
    not EXCLUDE_IF_MISSING
    or all(tp in [s.rstrip('*') for s in MEG_SELECTION] for tp in EXCLUDE_IF_MISSING)
    or (isinstance(MEG_SELECTION, str) and "all" in MEG_SELECTION.lower())
), f"Invalid EXCLUDE_IF_MISSING: {EXCLUDE_IF_MISSING}. Each entry must appear in MEG_SELECTION (ignoring '*') or MEG_SELECTION must contain 'all'."


### SET PATHS:
ROOT_DIR = Path(config['root_output_directory'])
DATA_INDEX_PATH = Path(ROOT_DIR) / 'master_data_catalogue.csv'
DATA_INDEX = pd.read_csv(DATA_INDEX_PATH)
PARAMETER_INDEX_PATH = Path(ROOT_DIR) / 'MEG_parameter_index.csv'
PARAMETER_INDEX = pd.read_csv(PARAMETER_INDEX_PATH)



# Check data catalogue to make sure QC metadata is present, if HAS_QC is set to 'True':
if HAS_QC:
    if (DATA_INDEX['n_good_MRIs'] == '[unknown]').any():
        raise Exception("QC metadata is expected ('skip_catalogue_QC' CONFIG parameter is set to 'False'), but master data catalogue appears to be missing QC metrics."
        f"\n\t--> Check 'master_data_catalogue.csv' in {ROOT_DIR} and ensure it has been successfully decorated with QC metadata.")

print("INPUT \ OUTPUT ===========================\n")
print(f"Main project directory set as: {ROOT_DIR}")
print()
print(f"Master data catalogue set as: {DATA_INDEX_PATH}")
print()
print(f"MEG parameter catalogue loaded from: {PARAMETER_INDEX_PATH}")
print()
print("SETTINGS ===========================\n")

print(f"OVERWRITE enabled: {MANIFEST_OVERWRITE}\n")

if HARD_STOP:
    print(f"Error mode: 'strict' (hard stops enabled)\n")
else:
    print(f"Error mode: 'soft' (hard stops disabled; errors will drop subjects)\n")

print(f"MRI analysis type: '{MRI_SELECTION}'\n")

if type(MEG_SELECTION) == list:
    print(f"MEG analysis type: Time-points\n  --> Study time-points selected for analysis: {MEG_SELECTION}\n")
elif type(MEG_SELECTION) == str:
    print(f"MEG analysis type: '{MEG_SELECTION}' [PRESET]\n")

print(f"QC metadata tables present?: {HAS_QC}")
if HAS_QC:
    print(f"  --> QC criteria for inclusion in analysis: {GOOD_LABELS}\n")

# Flexibly normalize 'CONDITIONS' parameter to either 'all' (str) or a list of group strings:
if isinstance(CONDITIONS, str):
    if CONDITIONS.strip().lower() == 'all':
        CONDITIONS = 'all'
        print("Analyzing 'all' subject groups (e.g. control + experimental) (as per config.yaml 'CONDITIONS' parameter)")
    else:
        raise Exception("ERROR: 'CONDITIONS' must be 'all' (plain string) or a LIST of group_ID strings.")
elif isinstance(CONDITIONS, list):
    CONDITIONS = [str(x).strip() for x in CONDITIONS]
    if len(CONDITIONS) == 1 and CONDITIONS[0].lower() == 'all':
        CONDITIONS = 'all'
        print("Analyzing 'all' subject groups (e.g. control + experimental) (as per config.yaml 'CONDITIONS' parameter)")
    elif any(x.lower() == 'all' for x in CONDITIONS):
        raise Exception("ERROR: If 'CONDITIONS' is a LIST, it cannot contain 'all' alongside other group names.")
    else:
        print(f"Analysis will include the following subject groups: {CONDITIONS}")
else:
    raise Exception("ERROR: 'CONDITIONS' must be either a string ('all') or a list of group_ID strings.")

--------

First, condition-based subsetting:

In [ ]:
print(f"Stage 1: Subsetting by experimental condition:")
print(f"\t-Selected conditions: {CONDITIONS}")
if isinstance(CONDITIONS, str) and CONDITIONS.lower() == 'all':
    SUBSET_1 = DATA_INDEX.copy()
else:
    # CONDITIONS is a list here:
    cond_lower = {c.lower() for c in CONDITIONS}
    available = set(DATA_INDEX['group_ID'].dropna().str.lower().unique())
    missing = sorted(cond_lower - available)
    if missing and HARD_STOP:
        raise ValueError(
            f"!!! ERROR: The following condition / subject group(s) were not found in 'group_ID': {missing}\n"
            "Check config.yaml 'CONDITIONS' settings.")
    if missing and not HARD_STOP:
        print(f"!!! WARNING: The following condition / subject group(s) are absent: {missing} — they will be ignored.\n")
    SUBSET_1 = DATA_INDEX[DATA_INDEX['group_ID'].str.lower().isin(cond_lower)]
    if SUBSET_1.empty:
        raise ValueError("After filtering by CONDITIONS, SUBSET is empty.")
SUBSET_1 = SUBSET_1.reset_index(drop=True).copy()
if DATA_INDEX.shape[0] != SUBSET_1.shape[0]:
    print(f"\n  --> Subsetted initial subject pool from {DATA_INDEX.shape[0]} --> {SUBSET_1.shape[0]} subjects:")
else:
    print(f"\n  --> No condition \ experimental group subsetting performed; proceeding from initial subject pool of {DATA_INDEX.shape[0]} subjects.")

# display(SUBSET_1)

## Note to self: maybe generate MRI_runs after ALL subsetting has been done? i.e. don't want to run MRI scans for subject_IDs that don't have valid 'baseline' MEG time points (assuming baseline is the main MEG criteria), etc... 

--------

MRI file selection:

In [ ]:
# Build MRI_runs from SUBSET_1 according to MRI_SELECTION
# Output columns: subject_ID, MRI_session_ID, MRI_filename

MRI_runs_rows = []

# Normalize MRI_SELECTION for mode detection
selection_normalized = str(MRI_SELECTION).strip().lower()

# Discover all MRI_*_filename columns and extract their session_IDs
session_id_pattern = re.compile(r"^MRI_(?P<session_ID>.+)_filename$")
available_session_IDs_with_MRI = []

for column in SUBSET_1.columns:
    match = session_id_pattern.match(column)
    if match:
        available_session_IDs_with_MRI.append(match.group("session_ID"))

if selection_normalized == "any":
    # Case 1: MRI_SELECTION == 'any' → collect ALL MRI_*_filename entries
    for _, row in SUBSET_1.iterrows():
        for session_ID in available_session_IDs_with_MRI:
            column_name = f"MRI_{session_ID}_filename"
            filename_value = row.get(column_name, pd.NA)

            if pd.notna(filename_value):
                filename_value = str(filename_value).strip()
                if filename_value:
                    MRI_runs_rows.append({
                        "subject_ID": row["subject_ID"],
                        "MRI_session_ID": session_ID,
                        "MRI_filename": filename_value})

else:
    # Case 2: MRI_SELECTION is treated as a specific target session_ID code
    target_session_ID = str(MRI_SELECTION).strip()
    target_column_name = f"MRI_{target_session_ID}_filename"

    # Hard error if this column does not exist
    if target_column_name not in SUBSET_1.columns:
        raise KeyError(
            f"Requested MRI_selection '{MRI_SELECTION}' does not correspond to a column "
            f"named '{target_column_name}' in SUBSET_1.")

    for _, row in SUBSET_1.iterrows():
        filename_value = row.get(target_column_name, pd.NA)

        if pd.notna(filename_value):
            filename_value = str(filename_value).strip()
            if filename_value:
                MRI_runs_rows.append({
                    "subject_ID": row["subject_ID"],
                    "MRI_session_ID": target_session_ID,
                    "MRI_filename": filename_value})

MRI_runs = pd.DataFrame(MRI_runs_rows)

# Optional quick sanity print if empty
if MRI_runs.empty:
    print("[info] MRI_runs is empty.")
    print(" - MRI_SELECTION (normalized):", selection_normalized)
    print(" - Available MRI timepoints (with filename columns):",
          available_session_IDs_with_MRI[:10],
          "..." if len(available_session_IDs_with_MRI) > 10 else "")

print(f"Original # of subjects in dataset: {SUBSET_1.shape[0]}")
print(f"Total number of valid MRI files found: {MRI_runs.shape[0]}")

# --- Deterministic per-subject selection when MRI_SELECTION == 'any' ---
# Chronology = TIME_POINTS_DICT key order (base session rank), then refine by run index for labels like '<base>-2'
if selection_normalized == "any" and not MRI_runs.empty:
    ordered_bases = list(TIME_POINTS_DICT.keys())
    base_to_rank = {label: i for i, label in enumerate(ordered_bases)}
    default_rank = len(ordered_bases) + 10**6  # push truly unknown bases to the end, deterministically

    def _parse_session_rank(session_id: str):
        s = str(session_id).strip()
        # Split "baseline-2" -> base="baseline", run=2 ; "UNKNOWN" -> base="UNKNOWN", run=1
        if "-" in s:
            base, maybe_idx = s.rsplit("-", 1)
            try:
                run_idx = int(maybe_idx)
            except ValueError:
                base, run_idx = s, 1
        else:
            base, run_idx = s, 1
        base_rank = base_to_rank.get(base, default_rank)
        return (base_rank, run_idx, base)

    MRI_runs["_session_rank"] = MRI_runs["MRI_session_ID"].apply(_parse_session_rank)

    # Sort so "earliest" is first, then keep first per subject deterministically
    MRI_runs = (
        MRI_runs
        .sort_values(["subject_ID", "_session_rank"], kind="stable")
        .drop_duplicates(subset=["subject_ID"], keep="first")
        .drop(columns=["_session_rank"])
        .reset_index(drop=True))
else:
    # --- Final de-duplication step: ensure only one MRI run per subject_ID ---
    MRI_runs = MRI_runs.drop_duplicates(subset=["subject_ID"], keep="first").reset_index(drop=True)

print(f"Total number of unique volumetric MRI reconstructions needed: {MRI_runs.shape[0]}")

MRI_runs

--------

"Re-harmonize" updated subject_IDs to SUBSET_2:

In [ ]:
# Re-harmonize to include only subjects that survived MRI selection
SUBSET_2 = SUBSET_1[SUBSET_1["subject_ID"].isin(MRI_runs["subject_ID"])].copy()

print(f"SUBSET_1: {SUBSET_1.shape[0]} subjects → SUBSET_2: {SUBSET_2.shape[0]} subjects after MRI filtering")

--------

Next, we perform MEG file selection:

In [ ]:
# =========================
# Build MEG_runs from SUBSET_2 according to MEG_SELECTION
# =========================

# Discover all MEG_*_filename columns dynamically
meg_pattern = re.compile(r"^MEG_(?P<session_ID>.+)_filename$")
available_MEG_session_IDs = []

for column in SUBSET_2.columns:
    match = meg_pattern.match(column)
    if match:
        available_MEG_session_IDs.append(match.group("session_ID"))

# Normalize the selection parameter
if isinstance(MEG_SELECTION, str) and MEG_SELECTION.strip().lower() == "all":
    selected_session_IDs = available_MEG_session_IDs[:]

elif isinstance(MEG_SELECTION, list):
    selected_session_IDs = [str(x).strip() for x in MEG_SELECTION]

    # Hard error if any requested column is missing
    missing = [
        sid for sid in selected_session_IDs
        if f"MEG_{sid}_filename" not in SUBSET_2.columns]
    if missing:
        raise KeyError(
            f"The following MEG session_IDs were requested in MEG_SELECTION but no corresponding "
            f"columns exist in SUBSET_2: {missing}")
else:
    raise ValueError("MEG_SELECTION must be either the string 'all' or a list of session_ID strings.")

# ---- Build wide-format table ----
subject_ids = SUBSET_2["subject_ID"].tolist()

columns = ["subject_ID"] + [f"MEG_{sid}_filename" for sid in selected_session_IDs]
MEG_runs = pd.DataFrame(index=range(len(subject_ids)), columns=columns)
MEG_runs["subject_ID"] = subject_ids

# Populate filename columns
for row_idx, (_, row) in enumerate(SUBSET_2.iterrows()):
    for session_ID in selected_session_IDs:
        column_name = f"MEG_{session_ID}_filename"
        filename_value = row.get(column_name, pd.NA)
        if pd.notna(filename_value):
            filename_value = str(filename_value).strip()
            if filename_value:
                MEG_runs.at[row_idx, column_name] = filename_value

# ---- Remove columns that are entirely empty ----
empty_cols = [
    column for column in MEG_runs.columns
    if column != "subject_ID" and MEG_runs[column].isna().all()]
if empty_cols:
    MEG_runs = MEG_runs.drop(columns=empty_cols)

# ---- Remove subjects with zero valid MEG files ----
data_cols = [col for col in MEG_runs.columns if col != "subject_ID"]

before_count = MEG_runs.shape[0]
MEG_runs = MEG_runs.dropna(subset=data_cols, how="all").reset_index(drop=True)
after_count = MEG_runs.shape[0]

dropped = before_count - after_count

print(f"[MEG_runs] wide format BEFORE filtering: {before_count} subjects")
print(f"[MEG_runs] Subjects dropped due to having no MEG files: {dropped}")
print(f"[MEG_runs] Final MEG_runs table: {after_count} subjects, {len(data_cols)} MEG session columns.\n")

MEG_runs.sample(7)

Filter down 'fMRI_runs' based on missing data tolerances & other analysis settings:

In [ ]:
# =========================
# Post-selection drops on MEG_runs + harmonize to SUBSET_3
# Rules:
# 1) Drop rows with no MEG files at all (all filename cols empty/NaN)  [always]
# 2) If COMPLETE_SUBS_ONLY == True → keep only rows where ALL filename cols are present
# 3) If EXCLUDE_IF_MISSING is a non-empty list of timepoints → drop rows missing any of those specific tps
# Finally: SUBSET_3 = SUBSET_2 restricted to surviving subject_IDs
# =========================

# Identify MEG filename columns (wide format)
filename_cols = [column for column in MEG_runs.columns
                 if column.startswith("MEG_") and column.endswith("_filename")]

if not filename_cols:
    raise ValueError("No MEG filename columns found in MEG_runs.")

# Helper: boolean DataFrame indicating non-empty filenames (case-insensitive, trims spaces)
_nonempty = MEG_runs[filename_cols].applymap(
    lambda value: (
        (isinstance(value, str) and value.strip() != "") or
        (pd.notna(value) and str(value).strip() != "")))

# --- (1) Hard drop: rows with zero files across all selected columns ---
drop_mask_allempty = ~_nonempty.any(axis=1)
n_drop_allempty = int(drop_mask_allempty.sum())
if n_drop_allempty:
    print(f"[drop] Subjects with no MEG files at all (given selection): {n_drop_allempty}")
    MEG_runs = MEG_runs.loc[~drop_mask_allempty].reset_index(drop=True)
    _nonempty = _nonempty.loc[~drop_mask_allempty].reset_index(drop=True)
else:
    print("[drop] No rows dropped for 'no MEG files at all'")

# --- (2) COMPLETE_SUBS_ONLY: keep only rows with all filename cols present ---
if COMPLETE_SUBS_ONLY:
    drop_mask_incomplete = ~_nonempty.all(axis=1)
    n_drop_incomplete = int(drop_mask_incomplete.sum())
    if n_drop_incomplete:
        print(f"[drop] COMPLETE_SUBS_ONLY == 'True' → subjects missing any selected MEG time-point: {n_drop_incomplete}")
        MEG_runs = MEG_runs.loc[~drop_mask_incomplete].reset_index(drop=True)
        _nonempty = _nonempty.loc[~drop_mask_incomplete].reset_index(drop=True)
    else:
        print("[drop] COMPLETE_SUBS_ONLY == 'True' → no subjects were incomplete")
else:
    print("[drop] COMPLETE_SUBS_ONLY == 'False' → no action")

# --- (3) EXCLUDE_IF_MISSING: list of mandatory timepoints to enforce ---
# Accepts falsy (None/False/[]/np.nan) → no action
if EXCLUDE_IF_MISSING:
    # Normalize to list of canonical timepoint keys present among filename_cols
    def _canon(s: str) -> str:
        return str(s).strip().lower().replace("_", " ")

    requested = list(EXCLUDE_IF_MISSING)

    # Build mapping from canonical session_ID to actual MEG filename column name
    session_to_column = {}
    for column in filename_cols:
        # column format: MEG_<session_ID>_filename
        session_ID = column[len("MEG_") : -len("_filename")]
        session_to_column[_canon(session_ID)] = column

    required_columns = [session_to_column[_canon(tp)] for tp in requested if _canon(tp) in session_to_column]

    # Warn (soft) if some requested mandatory timepoints are not in the selected columns
    missing_required = [tp for tp in requested if _canon(tp) not in session_to_column]
    if missing_required:
        print(f"[warn] EXCLUDE_IF_MISSING includes MEG timepoints not present in selection: {missing_required} (ignored)")

    if required_columns:
        # Drop rows where any required column is empty
        required_nonempty = _nonempty[required_columns]
        drop_mask_mand = ~required_nonempty.all(axis=1)
        n_drop_mand = int(drop_mask_mand.sum())
        if n_drop_mand:
            print(f"[drop] EXCLUDE_IF_MISSING enforced on {len(required_columns)} MEG timepoint(s) → dropped: {n_drop_mand}")
            MEG_runs = MEG_runs.loc[~drop_mask_mand].reset_index(drop=True)
            _nonempty = _nonempty.loc[~drop_mask_mand].reset_index(drop=True)
        else:
            print(f"[drop] EXCLUDE_IF_MISSING enforced on {len(required_columns)} MEG timepoint(s) → no rows dropped")
    else:
        print("[drop] EXCLUDE_IF_MISSING → no matching selected MEG columns; no action")
else:
    print("[drop] EXCLUDE_IF_MISSING not set → no action")

# --- Harmonize SUBSET_2 → SUBSET_3 based on surviving subject_IDs ---
surviving_subjects = MEG_runs["subject_ID"].dropna().unique().tolist()
SUBSET_3 = SUBSET_2[SUBSET_2["subject_ID"].isin(surviving_subjects)].reset_index(drop=True)

print(f"[SUBSET_3] After MEG-based filtering: {SUBSET_3.shape[0]} subjects (from {SUBSET_2.shape[0]}).")

In [ ]:
# --- Harmonize back to SUBSET_3 ---
SUBSET_3 = SUBSET_2[SUBSET_2["subject_ID"].isin(MEG_runs["subject_ID"])].copy()
print(f"[harmonize] SUBSET_2 → SUBSET_3: {SUBSET_2.shape[0]} → {SUBSET_3.shape[0]} subjects")
SUBSET_3.sample(10)

Next we "un-pack" the 'MEG_runs' dataframe so that we have one row for every file:

In [ ]:
# =========================
# Un-pack MEG_runs → one row per MEG file
# - Input:  MEG_runs (wide, one row per subject_ID; MEG_<session_ID>_filename cols)
# - Input:  SUBSET_3 (subject_ID-level metadata after MEG-based filtering)
# - Output: MEG_files: subject_ID, MEG_session_ID, MEG_filename, (+ any joined metadata from SUBSET_3)
# - Includes sanity check: count of non-empty filenames in MEG_runs == rows in MEG_files
# =========================

# 1) Build long-format rows: [subject_ID, MEG_session_ID, MEG_filename]
#    MEG_runs is always wide in the current MEG pipeline design.
filename_columns = [
    column for column in MEG_runs.columns
    if column.startswith("MEG_") and column.endswith("_filename")]

if not filename_columns:
    raise ValueError("No MEG filename columns found in MEG_runs.")

long_meg = MEG_runs.melt(
    id_vars=["subject_ID"],
    value_vars=filename_columns,
    var_name="wide_column",
    value_name="MEG_filename")

# Extract MEG_session_ID from column name: MEG_<session_ID>_filename
long_meg["MEG_session_ID"] = long_meg["wide_column"].str[len("MEG_") : -len("_filename")]

MEG_files = long_meg[["subject_ID", "MEG_session_ID", "MEG_filename"]].copy()

# 2) Clean: treat empty strings and trivial tokens as NaN, trim whitespace
MEG_files["MEG_filename"] = (
    MEG_files["MEG_filename"]
    .astype(str)
    .str.strip()
    .replace({"": np.nan, "NA": np.nan, "nan": np.nan}))

# 3) Drop rows with no filename (some subjects may have none after prior filters)
before = MEG_files.shape[0]
MEG_files = MEG_files.dropna(subset=["MEG_filename"]).reset_index(drop=True)
after = MEG_files.shape[0]
dropped_empty = before - after

if dropped_empty:
    print(f"[info] Dropped {dropped_empty} rows with empty/NaN MEG_filename during un-pack.")
else:
    print("[info] No empty MEG_filename rows encountered during un-pack.")

# === Sanity check: unpack consistency ===
# Count total non-empty filename entries in MEG_runs (across all *_filename cols)
def _is_nonempty(value):
    if pd.isna(value):
        return False
    string_value = str(value).strip()
    return string_value not in {"", "NA", "nan"}

nonempty_total = sum(
    MEG_runs[column].apply(_is_nonempty).sum()
    for column in filename_columns)
row_count = len(MEG_files)

assert nonempty_total == row_count, (
    f"Sanity check failed: total non-empty filenames in MEG_runs ({nonempty_total}) "
    f"≠ number of rows in MEG_files ({row_count}).")
print(
    f"[sanity-check] Non-empty MEG filenames in MEG_runs = {nonempty_total}  |  "
    f"Rows in MEG_files = {row_count}  [OK]")

# 4) (Optional but useful) join basic metadata from SUBSET_3
join_columns = ["subject_ID"]
if "group_ID" in SUBSET_3.columns:
    join_columns.append("group_ID")

metadata_for_join = SUBSET_3[join_columns].drop_duplicates(subset=["subject_ID"])

MEG_files = MEG_files.merge(
    metadata_for_join,
    how="left",
    on="subject_ID")

# 5) Sort for stable, human-readable ordering
MEG_files = MEG_files.sort_values(
    by=["subject_ID", "MEG_session_ID"]).reset_index(drop=True)

# 6) Reorder columns for readability
preferred_order = ["subject_ID", "group_ID", "MEG_session_ID", "MEG_filename"]
MEG_files = MEG_files[
    [column for column in preferred_order if column in MEG_files.columns]
    + [column for column in MEG_files.columns if column not in preferred_order]]

print(
    f"\nCompiled file list for {MEG_files.shape[0]} MEG files "
    f"(across {MEG_files['subject_ID'].nunique()} subject_IDs):")

# MEG_files

Should have everything we need now; let's compile the final dataframes and save / export:

In [ ]:
# =========================
# Merge MRI_runs + MEG_runs → run_manifest (anchor on MEG subjects)
# One row per subject_ID, with MRI info + MEG filename columns.
# =========================

# Collect MRI and MEG columns
mri_cols = [column for column in MRI_runs.columns if column.startswith("MRI_")]
meg_cols = [column for column in MEG_runs.columns if column.startswith("MEG_")]

# Reduce to subject_ID + MRI columns
if mri_cols:
    mri_df = MRI_runs[["subject_ID"] + mri_cols].copy()
else:
    mri_df = MRI_runs[["subject_ID"]].copy()

# Reduce to subject_ID + MEG columns
if meg_cols:
    meg_df = MEG_runs[["subject_ID"] + meg_cols].copy()
else:
    meg_df = MEG_runs[["subject_ID"]].copy()

# Merge anchored on MEG subjects (one row per subject_ID in MEG_runs)
run_manifest = meg_df.merge(
    mri_df,
    on="subject_ID",
    how="left",
    suffixes=("", "_MRIdup"))

# --- Add group_ID from SUBSET_3 (subject-level metadata) ---
if "group_ID" in SUBSET_3.columns:
    group_map = SUBSET_3[["subject_ID", "group_ID"]].drop_duplicates()
    run_manifest = run_manifest.merge(group_map, on="subject_ID", how="left")

# --- Reorder columns: subject_ID, group_ID, MRI..., MEG... ---
ordered_cols = ["subject_ID"]
if "group_ID" in run_manifest.columns:
    ordered_cols.append("group_ID")

ordered_cols += [column for column in run_manifest.columns if column in mri_cols]
ordered_cols += [column for column in run_manifest.columns if column in meg_cols]

run_manifest = run_manifest.reindex(columns=ordered_cols)

print(f"[run_manifest] {run_manifest.shape[0]} subjects; "
      f"{len(mri_cols)} MRI col(s), {len(meg_cols)} MEG col(s).")

if "group_ID" in run_manifest.columns:
    print(run_manifest.group_ID.value_counts())

# run_manifest.head(10)

--------

#### Final save / export:

In [ ]:
# =========================
# Export manifests (w/ conditional overwrite)
# =========================

if EXPORT_SUBSET:
    subset_export_filepath = Path(ROOT_DIR) / "run_data_catalog.csv"
    if not subset_export_filepath.exists() or MANIFEST_OVERWRITE:
        SUBSET_3.to_csv(subset_export_filepath, index=False)
        print(f"[export] Saved subset catalogue → {subset_export_filepath.name}")
    else:
        print(f"[skip] {subset_export_filepath.name} already exists; skipping (set MANIFEST_OVERWRITE=True to overwrite).")

run_manifest_filepath  = Path(ROOT_DIR) / "subject_manifest.csv"
MEG_manifest_filepath = Path(ROOT_DIR) / "MEG_manifest.csv"
for df, path in [(run_manifest, run_manifest_filepath), (MEG_files, MEG_manifest_filepath)]:
    if not path.exists() or MANIFEST_OVERWRITE:
        df.to_csv(path, index=False)
        print(f"[export] Saved {path.name}")
    else:
        print(f"[skip] {path.name} already exists; skipping (set MANIFEST_OVERWRITE=True to overwrite).")